# 01 — Данные
> **v2 | Агент удержания полосы + ПИД-регулятор | ветка: `v2-agent-pid`**

> 🕐 Обновлён: 2026-04-06 18:00 МСК

**Время:** ~5 минут  
Загружаем 3 000 сэмплов из Udacity, кэшируем в `data/cache.npy`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import sys, os

# Явно переходим в папку проекта
REPO_PATH = '/content/drive/MyDrive/diploma'
os.chdir(REPO_PATH)
sys.path.insert(0, REPO_PATH)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src.dataset_v2 import load_and_cache, angle_to_label, THRESHOLD, N_BINS

In [ ]:
# Распаковка датасета (если ещё не распакован)
import zipfile
ZIP_PATH = 'data/dataset.zip'
IMG_DIR  = 'data/IMG'

if not os.path.exists(IMG_DIR):
    print('Распаковываю...')
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall('data/')
    print('Готово')
else:
    print('Уже распакован')

# Ищем CSV
import glob
csvs = glob.glob('data/**/*.csv', recursive=True)
print('Найдены CSV:', csvs)

In [ ]:
CSV_PATH = csvs[0]   # первый найденный CSV
print(f'Используем: {CSV_PATH}')
df = pd.read_csv(CSV_PATH)
print(df.shape)
df.head(3)

In [ ]:
# images_dir = папка где лежит CSV (там же обычно IMG/)
import pathlib
IMAGES_DIR = str(pathlib.Path(CSV_PATH).parent) + '/'

# Загружаем и кэшируем 3k сэмплов
images, angles = load_and_cache(
    csv_path=CSV_PATH,
    images_dir=IMAGES_DIR,
    cache_path='data/cache.npy'
)
print(f'images: {images.shape}  angles: {angles.shape}')
print(f'Размер кэша в RAM: {images.nbytes / 1e6:.1f} MB')

In [ ]:
# ── Гистограмма углов до/после балансировки ──────────────────────
df_all = pd.read_csv(CSV_PATH)
angle_col = 'steering' if 'steering' in df_all.columns else df_all.columns[3]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df_all[angle_col], bins=50, color='steelblue', edgecolor='white')
axes[0].axvline(-THRESHOLD, color='red', linestyle='--', label=f'±{THRESHOLD}')
axes[0].axvline( THRESHOLD, color='red', linestyle='--')
axes[0].set_title(f'Полный датасет ({len(df_all):,} сэмплов)')
axes[0].set_xlabel('Угол руля')
axes[0].legend()

axes[1].hist(angles, bins=50, color='darkorange', edgecolor='white')
axes[1].axvline(-THRESHOLD, color='red', linestyle='--', label=f'±{THRESHOLD}')
axes[1].axvline( THRESHOLD, color='red', linestyle='--')
axes[1].set_title(f'После балансировки ({len(angles):,} сэмплов)')
axes[1].set_xlabel('Угол руля')
axes[1].legend()

plt.tight_layout()
plt.savefig('outputs/histograms.png', dpi=120)
plt.show()

In [ ]:
# ── Примеры изображений с метками ────────────────────────────────
labels = np.array([angle_to_label(a) for a in angles])
n_in  = labels.sum()
n_out = len(labels) - n_in
print(f'В полосе:   {n_in} ({100*n_in/len(labels):.1f}%)')
print(f'Вне полосы: {n_out} ({100*n_out/len(labels):.1f}%)')

fig, axes = plt.subplots(2, 6, figsize=(14, 5))
fig.suptitle('Примеры: зелёный = в полосе, красный = вне полосы')

for col, (label, color, title) in enumerate([
    (1, 'green', 'В полосе'),
    (0, 'red',   'Вне полосы')
]):
    idxs = np.where(labels == label)[0][:6]
    for row, i in enumerate(idxs):
        ax = axes[col][row]
        ax.imshow(images[i, 0], cmap='gray', vmin=-1, vmax=1)
        ax.set_title(f'e={angles[i]:.2f}', fontsize=8, color=color)
        ax.axis('off')

plt.tight_layout()
plt.savefig('outputs/sample_images.png', dpi=120)
plt.show()